In [6]:
import json, os, csv, glob, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import pearsonr
from transformers import AutoModel, AutoProcessor

MODEL_ID = "facebook/ijepa_vith14_22k"
BATCH = 8

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(MODEL_ID)
ijepa = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

with open("cutouts_split/split_manifest.json") as fh:
    manifest = json.load(fh)
train_files = [f"cutouts_split/train/{n}" for n in manifest["train"]]
test_files  = [f"cutouts_split/test/{n}"  for n in manifest["test"]]
print(len(train_files), len(test_files))

Loading weights: 100%|██████████| 517/517 [00:00<00:00, 5247.66it/s]

1600 400


In [7]:
def load_flux(p):
    g, r, z = np.asarray(np.load(p, allow_pickle=True).item()["flux"], dtype=np.float32)
    return np.stack([g, r, z])

# Platonic Universe scaling: per-channel 5th/99th percentiles over a large batch.
# Computed on TRAIN only and cached, so test embeddings can't shift with batch contents.
if os.path.exists("rgb_scale.npz"):
    s = np.load("rgb_scale.npz"); LO, HI = s["lo"], s["hi"]
else:
    sample = np.stack([load_flux(p) for p in train_files[:1000]])
    LO = np.percentile(sample, 5,  axis=(0, 2, 3), keepdims=True)
    HI = np.percentile(sample, 99, axis=(0, 2, 3), keepdims=True)
    np.savez("rgb_scale.npz", lo=LO, hi=HI)
print("lo", LO.ravel(), "hi", HI.ravel())


def to_rgb_uint8(flux):                             # flux: (3, H, W) as g, r, z
    s = np.clip((flux - LO[0]) / (HI[0] - LO[0] + 1e-8), 0, 1)
    rgb = np.stack([s[2], s[1], s[0]], axis=-1)     # z->R, r->G, g->B
    return (rgb * 255).astype(np.uint8)


def embed_ijepa(paths):
    out = []
    for i in range(0, len(paths), BATCH):
        batch = [to_rgb_uint8(load_flux(p)) for p in paths[i:i+BATCH]]
        inputs = processor(images=batch, return_tensors="pt").to(device)
        with torch.no_grad():
            emb = ijepa(**inputs).last_hidden_state.mean(dim=1)
        out.append(emb.cpu().numpy())
        print(f"{min(i+BATCH, len(paths))}/{len(paths)}", flush=True)
    return np.concatenate(out)


t = time.time()
probe_emb = embed_ijepa(train_files[:8])
per = (time.time() - t) / 8
print(f"{probe_emb.shape} | {per:.1f}s/image → {per*2000/3600:.1f} hours for 2000")

lo [-0.00283809 -0.00591219 -0.01720348] hi [0.03309599 0.07044824 0.11258927]
8/8
(8, 1280) | 3.3s/image → 1.8 hours for 2000
